## imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torch import fft

## helper functions

In [ ]:
def get_dataloaders(batch_size=128):
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408),   # ← CIFAR-100 stats
                             (0.2675, 0.2565, 0.2761)),
    ])

    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5071, 0.4867, 0.4408),
                             (0.2675, 0.2565, 0.2761)),
    ])

    trainset = torchvision.datasets.CIFAR100(
        root='./data', train=True, download=True, transform=transform_train)
    trainloader = torch.utils.data.DataLoader(
        trainset, batch_size=batch_size, shuffle=True,
        num_workers=6, pin_memory=True, persistent_workers=True)

    testset = torchvision.datasets.CIFAR100(
        root='./data', train=False, download=True, transform=transform_test)
    testloader = torch.utils.data.DataLoader(
        testset, batch_size=batch_size, shuffle=False,
        num_workers=6, pin_memory=True, persistent_workers=True)

    return trainloader, testloader


def train_cifar100(model, epochs=60, lr=0.001, weight_decay=5e-4, batch_size=128):
    """
    Modern minimal training loop for CIFAR-100
    Good starting point for most conv / ViT / ResNet / ConvNeXt / EfficientNetV2 models
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    trainloader, testloader = get_dataloaders(batch_size)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    print(f"Training on {device} — {len(trainloader.dataset)} train images, {len(testloader.dataset)} test")
    print(f"Batch size = {batch_size}, epochs = {epochs}, base LR = {lr}\n")

    best_acc = 0.0

    for epoch in range(epochs):
        model.train()
        correct = total = 0
        running_loss = 0.0

        for i, (images, labels) in enumerate(trainloader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            # stats
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            if (i + 1) % 100 == 0:
                print(f"[{epoch+1:2d}/{epochs}] [{i+1:5d}/{len(trainloader)}]  "
                      f"loss: {running_loss/100:.3f}  |  acc: {100.*correct/total:.2f}%")
                running_loss = correct = total = 0.0

        # ──── Evaluation ───────────────────────────────────────
        model.eval()
        correct = total = 0

        with torch.no_grad():
            for images, labels in testloader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                outputs = model(images)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        acc = 100. * correct / total
        print(f"Epoch {epoch+1:2d} ─ Test acc: {acc:.2f}%")

        if acc > best_acc:
            best_acc = acc
            print(f"↑ New best: {best_acc:.2f}%")

        scheduler.step()

    print(f"\nFinished. Best test accuracy: {best_acc:.2f}%")
    return model, best_acc

## architecture

### frequency amplifier

In [ ]:
class FreqAmp(nn.Module):
    """Amplifies underused freq components (low-magnitude ones)."""
    def __init__(self, amp_factor=2.0, thresh_quantile=0.3, dims=(-2, -1)):
        super().__init__()
        self.amp_factor = amp_factor  # how much to boost weak freq
        self.thresh_quantile = thresh_quantile  # e.g., boost bottom 30% magnitudes
        self.dims = dims

    def forward(self, x):
        z = fft.rfftn(x, dim=self.dims)  # real FFT for efficiency
        mag = torch.abs(z)

        # Compute threshold: e.g., 30th percentile of magnitudes
        # The error indicates that 'dim' must be an int, not a tuple in this PyTorch version.
        # To compute quantile across multiple dimensions, we flatten them first.

        # Flatten the dimensions specified in self.dims
        # For example, if mag is (B, C, H, W_rfft) and self.dims is (-2, -1),
        # mag_flat_for_quantile will be (B, C, H * W_rfft).
        mag_flat_for_quantile = mag.flatten(start_dim=self.dims[0], end_dim=self.dims[1])

        # Now apply quantile over the last dimension (the flattened one).
        # keepdim=True ensures the output has a dimension of 1 at the position
        # where quantile was applied, so thresh_flat_output will be (B, C, 1).
        thresh_flat_output = torch.quantile(mag_flat_for_quantile, self.thresh_quantile, dim=-1, keepdim=True)

        # To broadcast correctly with the original 'mag' (B, C, H, W_rfft),
        # 'thresh' needs to have a shape like (B, C, 1, 1). We add an extra
        # dimension of size 1 using unsqueeze.
        thresh = thresh_flat_output.unsqueeze(-1)

        # Mask for underused (weak) freq: amplify them
        weak_mask = (mag < thresh).float()
        scale = 1.0 + (self.amp_factor - 1.0) * weak_mask  # boost by amp_factor where weak

        z_amp = z * scale
        y = fft.irfftn(z_amp, dim=self.dims)  # back to spatial
        return y

### basic block

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()

        if stride != 1 or in_channels != out_channels:

            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

        self.freq_amp = FreqAmp(amp_factor=1.5, thresh_quantile=0.4)

    def forward(self, x):

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)

        identity = self.shortcut(x)
        itentity = self.freq_amp(identity)

        out += identity
        out = self.relu(out)
        return out

### FreqAmp block

In [ ]:
class FreqAmpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += self.shortcut(x)
        out = self.relu(out)
        return out

### model

In [ ]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=100, block_type='b'):
        super().__init__()
        if block_type == 'b':
            self.block_type = BasicBlock
        elif block_type == 'f':
            self.block_type = FreqAmpBlock
        else:
            raise ValueError(f"Invalid block type: {block_type}")

        self.in_channels = 64
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(self.block_type, 64, 2, stride=1)
        self.layer2 = self._make_layer(self.block_type, 128, 2, stride=2)
        self.layer3 = self._make_layer(self.block_type, 256, 2, stride=2)
        self.layer4 = self._make_layer(self.block_type, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)

        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        out = self.fc(out)
        return out

## interaction with models


### check architectures

In [ ]:
!pip install --upgrade onnx onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 12.2 MB/s eta 0:00:00


In [ ]:
torch_model = ResNet18(block_type='f')
# Create example inputs for exporting the model. The inputs should be a tuple of tensors.
example_inputs = (torch.randn(128, 3, 32, 32),)
onnx_program = torch.onnx.export(torch_model, example_inputs, dynamo=True)


/tmp/ipykernel_447/3456789514.py:4: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  onnx_program = torch.onnx.export(torch_model, example_inputs, dynamo=True)
W0305 17:07:54.413000 447 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0305 17:07:54.426000 447 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0305 17:07:54.429000 447 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_si

[torch.onnx] Obtain model graph for `ResNet18([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `ResNet18([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 60 of general pattern rewrite rules.


In [ ]:
onnx_program.save("ResNet18-FreqAmpBlocks.onnx")

### train basic model

In [ ]:
basic_block_resnet18 = ResNet18(block_type='b')

In [ ]:
trained_model, best_acc = train_cifar100(
    basic_block_resnet18,
    epochs=120,           # 60–200 depending on model size
    lr=0.001,             # 0.0005–0.003 common
    weight_decay=5e-4,
    batch_size=256        # increase if you have enough VRAM
)
torch.save(trained_model.state_dict(), 'basic_block_resnet18')

Training on cuda — 50000 train images, 10000 test
Batch size = 256, epochs = 120, base LR = 0.001

[ 1/120] [  100/196]  loss: 4.154  |  acc: 8.04%
Epoch  1 ─ Test acc: 16.40%
↑ New best: 16.40%
[ 2/120] [  100/196]  loss: 3.526  |  acc: 20.10%
Epoch  2 ─ Test acc: 27.67%
↑ New best: 27.67%
[ 3/120] [  100/196]  loss: 3.094  |  acc: 30.20%
Epoch  3 ─ Test acc: 35.38%
↑ New best: 35.38%
[ 4/120] [  100/196]  loss: 2.758  |  acc: 39.07%
Epoch  4 ─ Test acc: 39.41%
↑ New best: 39.41%
[ 5/120] [  100/196]  loss: 2.523  |  acc: 45.91%
Epoch  5 ─ Test acc: 45.81%
↑ New best: 45.81%
[ 6/120] [  100/196]  loss: 2.348  |  acc: 50.59%
Epoch  6 ─ Test acc: 46.93%
↑ New best: 46.93%
[ 7/120] [  100/196]  loss: 2.196  |  acc: 55.35%
Epoch  7 ─ Test acc: 51.05%
↑ New best: 51.05%
[ 8/120] [  100/196]  loss: 2.060  |  acc: 59.48%
Epoch  8 ─ Test acc: 54.67%
↑ New best: 54.67%
[ 9/120] [  100/196]  loss: 1.938  |  acc: 63.50%
Epoch  9 ─ Test acc: 56.86%
↑ New best: 56.86%
[10/120] [  100/196]  loss: 1

### train freqamp block model

In [ ]:
freqamp_block_resnet18 = ResNet18(block_type='f')

In [ ]:
trained_model, best_acc = train_cifar100(
    freqamp_block_resnet18,
    epochs=120,           # 60–200 depending on model size
    lr=0.001,             # 0.0005–0.003 common
    weight_decay=5e-4,
    batch_size=256        # increase if you have enough VRAM
)
torch.save(trained_model.state_dict(), 'freqamp_block_resnet18')

Training on cuda — 50000 train images, 10000 test
Batch size = 256, epochs = 120, base LR = 0.001

[ 1/120] [  100/196]  loss: 4.140  |  acc: 8.12%
Epoch  1 ─ Test acc: 15.02%
↑ New best: 15.02%
[ 2/120] [  100/196]  loss: 3.414  |  acc: 22.55%
Epoch  2 ─ Test acc: 27.82%
↑ New best: 27.82%
[ 3/120] [  100/196]  loss: 2.973  |  acc: 33.35%
Epoch  3 ─ Test acc: 33.31%
↑ New best: 33.31%
[ 4/120] [  100/196]  loss: 2.668  |  acc: 41.09%
Epoch  4 ─ Test acc: 39.67%
↑ New best: 39.67%
[ 5/120] [  100/196]  loss: 2.456  |  acc: 47.80%
Epoch  5 ─ Test acc: 46.98%
↑ New best: 46.98%
[ 6/120] [  100/196]  loss: 2.275  |  acc: 52.92%
Epoch  6 ─ Test acc: 47.87%
↑ New best: 47.87%
[ 7/120] [  100/196]  loss: 2.131  |  acc: 57.45%
Epoch  7 ─ Test acc: 53.29%
↑ New best: 53.29%
[ 8/120] [  100/196]  loss: 2.020  |  acc: 60.64%
Epoch  8 ─ Test acc: 54.47%
↑ New best: 54.47%
[ 9/120] [  100/196]  loss: 1.899  |  acc: 64.41%
Epoch  9 ─ Test acc: 58.57%
↑ New best: 58.57%
[10/120] [  100/196]  loss: 1

KeyboardInterrupt: 

## quick tests